# Building the Modeling Dataset

Joins the daily weather table (`merra2_pipeline.ipynb`) to the daily AQI series
(`aqi_pipeline.ipynb`), engineers lagged and seasonal features, and writes a
fixed chronological train/test split that all six model notebooks share.

**Prediction task:** same-day AQI from same-day weather plus the previous three
days of weather. LA's bad-air episodes build over several days of heat and
stagnation, so yesterday's conditions carry real information about today.

Outputs:
- `data/processed/la_modeling_dataset.csv` — full joined + engineered table
- `data/processed/train.csv` / `test.csv` — the split every model uses

## Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

PROCESSED = ROOT / 'data' / 'processed'

WEATHER_CSV = PROCESSED / 'la_daily_weather_2016_2025.csv'
AQI_CSV     = PROCESSED / 'la_daily_aqi_5pollutants_v2_2016_2025.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

weather = pd.read_csv(WEATHER_CSV, parse_dates=['date'])
aqi     = pd.read_csv(AQI_CSV,     parse_dates=['date'])

print(f'weather: {weather.shape}  {weather.date.min().date()} -> {weather.date.max().date()}')
print(f'aqi:     {aqi.shape}  {aqi.date.min().date()} -> {aqi.date.max().date()}')

weather: (3652, 14)  2016-01-01 -> 2025-12-30
aqi:     (3653, 7)  2016-01-01 -> 2025-12-31


## Joining the two series

The AQI series runs one day longer than the weather series — MERRA-2 had not
published 2025-12-31 at the time of the download. An inner join drops that single
day rather than carrying a row with no features.

In [2]:
df = weather.merge(aqi[['date', 'daily_aqi', 'dominant_pollutant']],
                   on='date', how='inner')

print(f'joined: {len(df)} rows  ({len(aqi) - len(df)} AQI day(s) dropped for lack of weather)')
print(f'range:  {df.date.min().date()} -> {df.date.max().date()}')
print(f'missing values: {df.isna().sum().sum()}')

joined: 3652 rows  (1 AQI day(s) dropped for lack of weather)
range:  2016-01-01 -> 2025-12-30
missing values: 0


## Encoding the circular features

Two variables wrap around and can't be fed to a model as raw numbers.

Wind direction is the obvious one: 359° and 1° are neighbours, but numerically
they sit at opposite ends of the range, so a linear model would read a two-degree
shift as a 358-unit jump. Day-of-year has the same problem at the New Year
boundary. Both get split into sine and cosine components, which preserves the
wrap-around.

In [3]:
# wind direction -> unit vector
rad = np.deg2rad(df['wind_dir_mean'])
df['wind_dir_sin'] = np.sin(rad)
df['wind_dir_cos'] = np.cos(rad)
df = df.drop(columns=['wind_dir_mean'])

# day of year -> unit vector (captures season without a hard January cliff)
doy = df['date'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)

# weekday effects: LA traffic drops on weekends, and NOx with it
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)

print(df[['wind_dir_sin', 'wind_dir_cos', 'doy_sin', 'doy_cos', 'is_weekend']].describe().T.to_string())

               count      mean       std       min       25%       50%       75%       max
wind_dir_sin  3652.0 -0.475849  0.604443 -1.000000 -0.931832 -0.767281 -0.139100  0.999982
wind_dir_cos  3652.0 -0.085559  0.633329 -1.000000 -0.603855 -0.260414  0.531897  0.999985
doy_sin       3652.0  0.000007  0.707252 -0.999999 -0.703677 -0.004301  0.706727  0.999986
doy_cos       3652.0 -0.000137  0.707155 -0.999979 -0.707487  0.001075  0.702913  0.999991
is_weekend    3652.0  0.285871  0.451890  0.000000  0.000000  0.000000  1.000000  1.000000


## Adding lagged and rolling features

Five variables get 1-, 2-, and 3-day lags plus a 3-day rolling mean. These are the
ones with a physical reason to persist: heat builds, moisture lingers, and a
stalled air mass stays stalled.

The rolling mean is computed on the trailing window **including** the current day,
which is legitimate here — we're predicting today's AQI from today's weather, so
today's weather is an input, not a leak.

In [4]:
LAG_VARS = ['t2m_mean', 't2m_max', 'qv2m_mean', 'wind_speed_mean', 'calm_hours']
LAGS = [1, 2, 3]

for var in LAG_VARS:
    for L in LAGS:
        df[f'{var}_lag{L}'] = df[var].shift(L)
    df[f'{var}_roll3'] = df[var].rolling(window=3, min_periods=3).mean()

before = len(df)
df = df.dropna().reset_index(drop=True)   # drops the first 2 days, which have no full window
print(f'dropped {before - len(df)} warm-up rows -> {len(df)} usable rows')
print(f'range: {df.date.min().date()} -> {df.date.max().date()}')
print(f'total columns: {df.shape[1]}')

dropped 3 warm-up rows -> 3649 usable rows
range: 2016-01-04 -> 2025-12-30
total columns: 40


## Which weather features actually track AQI?

A correlation check before modeling — partly a sanity test, partly to know what to
expect from the linear model.

In [5]:
FEATURES = [c for c in df.columns if c not in ('date', 'daily_aqi', 'dominant_pollutant')]

corr = (df[FEATURES + ['daily_aqi']].corr()['daily_aqi']
        .drop('daily_aqi').sort_values(key=abs, ascending=False))
print(f'{len(FEATURES)} features\n')
print('Top 12 by |correlation| with daily AQI:')
print(corr.head(12).round(3).to_string())
print('\nWeakest 5:')
print(corr.tail(5).round(3).to_string())

37 features

Top 12 by |correlation| with daily AQI:
t2m_max           0.705
t2m_mean          0.693
t2m_max_roll3     0.678
t2m_max_lag1      0.671
t2m_mean_roll3    0.650
t2m_mean_lag1     0.641
t2m_min           0.639
t2m_max_lag2      0.601
t2m_mean_lag2     0.578
t2m_max_lag3      0.542
t2m_mean_lag3     0.531
t2m_range         0.478

Weakest 5:
wind_dir_sin   -0.183
wind_dir_cos   -0.134
v10m_mean       0.070
is_weekend      0.043
u10m_mean      -0.008


Temperature dominates: `t2m_max` correlates at **0.705** with daily AQI, and the
next four spots are all temperature variants. That is the ozone story — LA's AQI
is driven by photochemical smog, which needs heat.

The stagnation signal shows up right behind it. `wind_speed_mean` runs at −0.415
and `calm_hours` at +0.442, pointing the same direction from both sides: moving
air disperses pollution, still air traps it. Notably the **3-day rolling versions
beat the same-day values** for both (`calm_hours_roll3` 0.466 vs `calm_hours`
0.442), which is direct evidence that these are multi-day episodes and justifies
carrying the lag features.

At the bottom, `u10m_mean` (−0.008) and `is_weekend` (0.043) are essentially
uncorrelated. The weekend result is a little surprising given LA traffic, but the
city-wide AQI is usually ozone-driven, and weekend ozone can actually *rise* when
NOx falls — the two effects likely cancel.

A single feature at 0.705 means a linear model will capture the main trend, but
the relationship between heat, stagnation and smog is a threshold effect rather
than a straight line, so there should be real headroom for the nonlinear models.

## Splitting train and test chronologically

This is a time series, so the split has to respect time. A random split would let
the model train on 2024 and test on 2023 — it would see the future, and the test
score would be optimistic in a way that means nothing.

Train on 2016–2022, test on 2023–2025. That's roughly 70/30, and it asks the
honest question: fit on seven years of history, then predict three years the model
has never seen.

In [6]:
SPLIT_DATE = pd.Timestamp('2023-01-01')

train = df[df.date <  SPLIT_DATE].reset_index(drop=True)
test  = df[df.date >= SPLIT_DATE].reset_index(drop=True)

print(f'train: {len(train):5} rows  {train.date.min().date()} -> {train.date.max().date()}')
print(f'test:  {len(test):5} rows  {test.date.min().date()} -> {test.date.max().date()}')
print(f'split: {len(train)/len(df):.1%} / {len(test)/len(df):.1%}')

print('\nTarget distribution:')
print(pd.DataFrame({'train': train.daily_aqi.describe(),
                    'test':  test.daily_aqi.describe()}).round(2).to_string())

train:  2554 rows  2016-01-04 -> 2022-12-31
test:   1095 rows  2023-01-01 -> 2025-12-30
split: 70.0% / 30.0%

Target distribution:
         train     test
count  2554.00  1095.00
mean     89.37    87.34
std      37.01    40.39
min      31.00    37.00
25%      62.00    57.00
50%      77.00    72.00
75%     108.00   108.00
max     250.00   230.00


The test period runs a little cleaner than the training period — a real trend in
LA air quality, not an artifact. Worth remembering when reading the results: every
model is being asked to extrapolate slightly downward.

## Saving

The three CSVs plus a small JSON manifest, so the six model notebooks all load the
identical split and feature list instead of each rebuilding it.

In [7]:
df.to_csv(PROCESSED / 'la_modeling_dataset.csv', index=False)
train.to_csv(PROCESSED / 'train.csv', index=False)
test.to_csv(PROCESSED / 'test.csv',  index=False)

manifest = {
    'target': 'daily_aqi',
    'features': FEATURES,
    'n_features': len(FEATURES),
    'split_date': str(SPLIT_DATE.date()),
    'n_train': len(train),
    'n_test': len(test),
    'train_range': [str(train.date.min().date()), str(train.date.max().date())],
    'test_range':  [str(test.date.min().date()),  str(test.date.max().date())],
}
with open(PROCESSED / 'feature_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Saved:')
for name in ['la_modeling_dataset.csv', 'train.csv', 'test.csv', 'feature_manifest.json']:
    print(f'  data/processed/{name}')
print(f'\n{manifest["n_features"]} features, target = {manifest["target"]}')
df.head()

Saved:
  data/processed/la_modeling_dataset.csv
  data/processed/train.csv
  data/processed/test.csv
  data/processed/feature_manifest.json

37 features, target = daily_aqi


,date,t2m_mean,t2m_max,t2m_min,t2m_range,qv2m_mean,qv2m_max,u10m_mean,v10m_mean,wind_speed_mean,wind_speed_max,wind_speed_min,calm_hours,daily_aqi,dominant_pollutant,wind_dir_sin,wind_dir_cos,doy_sin,doy_cos,is_weekend,t2m_mean_lag1,t2m_mean_lag2,t2m_mean_lag3,t2m_mean_roll3,t2m_max_lag1,t2m_max_lag2,t2m_max_lag3,t2m_max_roll3,qv2m_mean_lag1,qv2m_mean_lag2,qv2m_mean_lag3,qv2m_mean_roll3,wind_speed_mean_lag1,wind_speed_mean_lag2,wind_speed_mean_lag3,wind_speed_mean_roll3,calm_hours_lag1,calm_hours_lag2,calm_hours_lag3,calm_hours_roll3
0,2016-01-04,11.156650,14.856079,8.998993,5.857086,6.437552,7.229472,-3.336517,0.726196,3.842359,5.233736,1.065249,4,68.0,PM2.5,0.977124,-0.212672,0.068755,0.997634,0,10.133282,9.579383,7.802708,10.289772,14.810822,15.884277,14.086761,15.183726,5.310264,3.442668,2.930061,5.063495,3.313107,2.998282,4.326950,3.384583,4.0,2.0,4.0,3.333333
1,2016-01-05,11.257178,13.084229,10.060364,3.023865,7.811948,8.766274,0.448972,3.415153,4.437946,9.754294,1.011675,9,58.0,PM2.5,-0.130343,-0.991469,0.085906,0.996303,0,11.156650,10.133282,9.579383,10.849037,14.856079,14.810822,15.884277,14.250376,6.437552,5.310264,3.442668,6.519921,3.842359,3.313107,2.998282,3.864471,4.0,4.0,2.0,5.666667
2,2016-01-06,9.204604,11.935822,7.217926,4.717896,6.876637,8.120941,4.390279,1.726155,5.373285,9.036048,2.883062,0,66.0,PM2.5,-0.930650,-0.365910,0.103031,0.994678,0,11.257178,11.156650,10.133282,10.539477,13.084229,14.856079,14.810822,13.292043,7.811948,6.437552,5.310264,7.042046,4.437946,3.842359,3.313107,4.551197,9.0,4.0,4.0,4.333333
3,2016-01-07,9.084325,10.674194,7.529083,3.145111,6.364306,7.234623,4.513538,1.206327,5.113807,6.947113,4.085638,0,62.0,PM2.5,-0.966090,-0.258205,0.120126,0.992759,0,9.204604,11.257178,11.156650,9.848702,11.935822,13.084229,14.856079,11.898081,6.876637,7.811948,6.437552,7.017630,5.373285,4.437946,3.842359,4.975013,0.0,9.0,4.0,3.000000
4,2016-01-08,7.964045,13.524963,5.229217,8.295746,5.172918,5.808143,2.431457,-2.458621,3.680065,4.737813,2.653392,0,64.0,PM2.5,-0.703168,0.711024,0.137185,0.990545,0,9.084325,9.204604,11.257178,8.750991,10.674194,11.935822,13.084229,12.044993,6.364306,6.876637,7.811948,6.137953,5.113807,5.373285,4.437946,4.722386,0.0,0.0,9.0,0.000000
